# Configuration env

In [1]:
%load_ext sql
%config SqlMagic.autopandas = True
%sql duckdb:///:memory:
%sql ATTACH IF NOT EXISTS 'data/risk.db' AS r (TYPE sqlite);

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

,Success


# Analyse des tables

In [4]:
%%sql
select database, name, column_names, column_types from (show all tables) where database = 'r';

Running query in 'duckdb:///:memory:'

,database,name,column_names,column_types
0,r,mkt_forward_curve,"[as_of_date, commodity, delivery_month, price_...","[VARCHAR, VARCHAR, VARCHAR, DOUBLE]"
1,r,mkt_spot_hourly,"[commodity, delivery_hour_utc, price_eur_mwh]","[VARCHAR, VARCHAR, DOUBLE]"
2,r,pos_snapshot,"[commodity, delivery_month, book, source_syste...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, DOUBLE, V..."
3,r,ref_contract,"[contract_id, customer_id, commodity, start_da...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ..."
4,r,ref_customer,"[customer_id, customer_name, sector, segment, ...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR]"
5,r,ref_site,"[site_id, customer_id, commodity, region, dso,...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ..."
6,r,trd_deal,"[deal_id, trade_date, trade_ts, commodity, dir...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ..."


In [5]:
%%sql 
select database_name, table_name, column_count, estimated_size as n_lignes from duckdb_tables()
where database_name = 'r'
order by n_lignes desc;

Running query in 'duckdb:///:memory:'

,database_name,table_name,column_count,n_lignes
0,r,mkt_spot_hourly,3,17519
1,r,trd_deal,13,9579
2,r,mkt_forward_curve,4,9381
3,r,ref_site,8,1399
4,r,pos_snapshot,6,473
5,r,ref_contract,7,259
6,r,ref_customer,5,219


## Analyse trd_deal

In [7]:
%%sql
select * from (describe r.trd_deal);

Running query in 'duckdb:///:memory:'

,column_name,column_type,null,key,default,extra
0,deal_id,VARCHAR,YES,None,None,None
1,trade_date,VARCHAR,YES,None,None,None
2,trade_ts,VARCHAR,YES,None,None,None
3,commodity,VARCHAR,YES,None,None,None
4,direction,VARCHAR,YES,None,None,None
5,delivery_start,VARCHAR,YES,None,None,None
6,delivery_end,VARCHAR,YES,None,None,None
7,volume_mwh,DOUBLE,YES,None,None,None
8,price_eur_mwh,DOUBLE,YES,None,None,None
9,counterparty,VARCHAR,YES,None,None,None


In [8]:
%%sql
select * from (summarize r.trd_deal);

Running query in 'duckdb:///:memory:'

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,deal_id,VARCHAR,D2600000,D2608999,13090,None,None,None,None,None,9580,0.0
1,trade_date,VARCHAR,2025-06-02,2026-07-24,367,None,None,None,None,None,9580,0.0
2,trade_ts,VARCHAR,2025-06-02 08:04:13,2026-07-25 13:16:03,8677,None,None,None,None,None,9580,0.0
3,commodity,VARCHAR,GAS,POWER,2,None,None,None,None,None,9580,0.0
4,direction,VARCHAR,BUY,SELL,2,None,None,None,None,None,9580,0.0
5,delivery_start,VARCHAR,2026-01-01,2027-12-01,23,None,None,None,None,None,9580,0.0
6,delivery_end,VARCHAR,2026-01-31,2028-11-30,36,None,None,None,None,None,9580,0.0
7,volume_mwh,DOUBLE,9.7,6910.4,4787,314.23159707724534,318.7419023861352,123.5933455335743,220.37963555555552,388.09233592031393,9580,0.0
8,price_eur_mwh,DOUBLE,15.694,124.153,8619,59.38184582463463,26.113370030476737,32.97382089376254,63.01423419606519,81.31263836442038,9580,0.0
9,counterparty,VARCHAR,AXPO,VITOL,12,None,None,None,None,None,9580,0.0


In [12]:
%%sql 
select count(*) as n, 
count(distinct deal_id) as nd
from r.trd_deal;

Running query in 'duckdb:///:memory:'

,n,nd
0,9580,9000


In [19]:
%%sql 
select deal_id, count(deal_id) as n_deal
from r.trd_deal
group by deal_id
having count(deal_id) > 1


Running query in 'duckdb:///:memory:'

,deal_id,n_deal
0,D2600011,2
1,D2600374,2
2,D2600500,2
3,D2600562,2
4,D2600967,2
...,...,...
570,D2608284,2
571,D2608612,2
572,D2608636,2
573,D2608752,2


In [20]:
%%sql 
select * from (
    select deal_id, count(deal_id) as n_deal
    from r.trd_deal
    group by deal_id
    having count(deal_id) > 1
    )
where n_deal = 3

Running query in 'duckdb:///:memory:'

,deal_id,n_deal
0,D2601497,3
1,D2601680,3
2,D2600313,3
3,D2601976,3
4,D2608057,3


In [12]:
%%sql
SELECT COUNT(*)
FROM (
    SELECT *
    FROM r.trd_deal
    GROUP BY ALL
    HAVING COUNT(*) > 1
);

Running query in 'duckdb:///:memory:'

,count_star()
0,40


In [19]:
%%sql
SELECT *
FROM r.trd_deal
GROUP BY ALL
HAVING COUNT(*) > 1;

Running query in 'duckdb:///:memory:'

,deal_id,trade_date,trade_ts,commodity,direction,delivery_start,delivery_end,volume_mwh,price_eur_mwh,counterparty,book,status,version
0,D2604870,2025-07-21,2025-07-21 13:02:20,POWER,SELL,2026-02-01,2026-02-28,431.9,89.913,EDF_TRADING,B2B_FR_GAS_TRADING,CONFIRMED,1
1,D2608645,2026-05-21,2026-05-21 10:08:23,POWER,BUY,2026-12-01,2026-12-31,110.0,97.873,SHELL_ENERGY,B2B_FR_POWER_HEDGE,CONFIRMED,1
2,D2604243,2026-06-04,2026-06-04 11:52:55,GAS,BUY,2026-01-01,2026-01-31,474.1,37.518,GUNVOR,B2B_FR_GAS_HEDGE,CONFIRMED,1
3,D2606526,2025-07-07,2025-07-07 11:22:00,POWER,BUY,2026-06-01,2026-06-30,73.5,66.594,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,1
4,D2600890,2025-07-31,2025-07-31 17:23:36,GAS,BUY,2026-12-01,2026-12-31,96.3,36.701,SHELL_ENERGY,B2B_FR_GAS_HEDGE,CONFIRMED,1
5,D2605509,2026-01-30,2026-01-30 13:27:34,GAS,BUY,2026-02-01,2026-02-28,394.6,40.141,RWE,B2B_FR_GAS_HEDGE,CONFIRMED,1
6,D2607350,2026-03-06,2026-03-06 09:11:02,GAS,BUY,2026-06-01,2026-08-31,695.1,25.514,STATKRAFT,B2B_FR_GAS_HEDGE,CONFIRMED,1
7,D2607639,2025-11-05,2025-11-05 13:36:28,GAS,BUY,2027-01-01,2027-01-31,98.7,41.196,UNIPER,B2B_FR_GAS_HEDGE,CONFIRMED,1
8,D2608303,2026-06-03,2026-06-03 14:23:14,GAS,BUY,2026-09-01,2026-09-30,327.0,30.684,VITOL,B2B_FR_POWER_TRADING,CONFIRMED,1
9,D2601543,2025-07-31,2025-07-31 17:04:32,POWER,BUY,2027-11-01,2027-11-30,523.7,87.376,MERCURIA,B2B_FR_POWER_HEDGE,CONFIRMED,1
